# import

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully")

Libraries imported successfully


In [54]:
DATA_PATH = r"D:\EY_Data_Science_Challenge\data\raw\Challenge_Data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

Dataset loaded successfully
Rows    : 5,899
Columns : 8


# Data info

In [55]:
df.info()             

<class 'pandas.DataFrame'>
RangeIndex: 5899 entries, 0 to 5898
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Col1                 5899 non-null   str  
 1   Col2                 5899 non-null   str  
 2   Col3                 5899 non-null   str  
 3   Col4                 5746 non-null   str  
 4   Col5                 5899 non-null   str  
 5   Col6                 5899 non-null   str  
 6   Col7                 5899 non-null   str  
 7   ClassificationLabel  5899 non-null   str  
dtypes: str(8)
memory usage: 1017.0 KB


###### col4 has missing values
###### input feature 7, target is ClassificationLabel

In [56]:
df.columns

Index(['Col1', 'Col2', 'Col3', 'Col4', 'Col5', 'Col6', 'Col7',
       'ClassificationLabel'],
      dtype='str')

In [57]:
df.isnull().sum()

Col1                     0
Col2                     0
Col3                     0
Col4                   153
Col5                     0
Col6                     0
Col7                     0
ClassificationLabel      0
dtype: int64

In [58]:
# Calculate the percentage of missing values
df.isnull().mean() * 100

Col1                   0.00000
Col2                   0.00000
Col3                   0.00000
Col4                   2.59366
Col5                   0.00000
Col6                   0.00000
Col7                   0.00000
ClassificationLabel    0.00000
dtype: float64

In [59]:
# Count duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 674


In [60]:
# Check the number of unique values in each column
df.nunique()

Col1                    313
Col2                   4818
Col3                   2397
Col4                   2032
Col5                     17
Col6                    196
Col7                      4
ClassificationLabel      11
dtype: int64

In [61]:
# Count the number of records in each class
df["ClassificationLabel"].value_counts()

ClassificationLabel
Category_1     5204
Category_2      627
Category4        16
Category_3       12
category_1       11
Categry_6        10
Category 3       10
Category2         4
Category_6        2
Category 5        2
Category _3       1
Name: count, dtype: int64

## Target Label Quality Check

During the target variable analysis, some inconsistencies were found in the
classification labels.

Examples include:

- `category_1` vs `Category_1`
- `Category2` vs `Category_2`
- `Category 3` vs `Category_3`
- `Category _3` vs `Category_3`
- `Categry_6` vs `Category_6`

These appear to be formatting or spelling inconsistencies rather than
different business classes. Before model training, these labels should be
standardized.

The original raw dataset will not be modified.

###### Category_1 alone represents about 88.4% of the data, while Category_5 has only 2 records.
###### This is going to be an important modeling challenge. We should not use accuracy alone because a model predicting mostly Category_1 could still achieve very high accuracy.
###### Also, the classes with only 2, 12, and 16 samples create a problem for a normal stratified train/test split and especially for cross-validation. We'll handle that carefully later.

### Feature Analysis

In [62]:
# Check the values in each feature

for col in df.columns[:7]:
    print("\nColumn:", col)
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(10))


Column: Col1
Unique values: 313
Col1
Word16 Word17 Word18 Word7                                     1245
Word8 Word9 Word10 Word11 Word5                                 323
Word30 Word31                                                   264
Word19 Word20 Word21 Word22 Word23 Word5                        203
Word16 Word17 Word3 Word7                                       198
Word85 Word86 Word87 Word5 Word88 Word16 Word17 Word3 Word7     196
Word33 Word34 Word35 Word7                                      173
Word42 Word43 Word5                                             163
Word38 Word4 Word39                                             161
Word96 Word3 Word4 Word5                                        141
Name: count, dtype: int64

Column: Col2
Unique values: 4818
Col2
201811-00046    59
201806-00045    35
4.80P+11        22
4.80W+11        19
4.80J+11        19
4.80F+11        17
4.80U+11        17
4.80Q+11        14
4.80I+11        14
4.80R+11        14
Name: count, dtype: int64



In [63]:
# Check whether the features contain numeric values

for col in df.columns[:7]:
    numeric = pd.to_numeric(df[col], errors="coerce")
    percentage = numeric.notna().mean() * 100
    
    print(col, ":", round(percentage, 2), "% numeric")

Col1 : 0.0 % numeric
Col2 : 60.01 % numeric
Col3 : 46.21 % numeric
Col4 : 0.0 % numeric
Col5 : 0.0 % numeric
Col6 : 0.0 % numeric
Col7 : 0.0 % numeric


In [64]:
# Check whether the columns contain date-like values

for col in df.columns[:7]:
    dates = pd.to_datetime(df[col], errors="coerce", utc=True)
    percentage = dates.notna().mean() * 100

    print(col, ":", round(percentage, 2), "% date-like")

Col1 : 0.0 % date-like


C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[col], errors="coerce", utc=True)


Col2 : 24.41 % date-like
Col3 : 0.0 % date-like
Col4 : 0.0 % date-like
Col5 : 67.96 % date-like
Col6 : 0.0 % date-like
Col7 : 0.0 % date-like


C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\victus\AppData\Local\Temp\ipykernel_17884\1630050695

In [65]:
for col in df.columns[:7]:
    print("\n", col)
    print(df[col].head(5).tolist())


 Col1
['Word1 Word2 Word3 Word4 Word5', 'Word6 Word7', 'Word8 Word9 Word10 Word11 Word5', 'Word12 Word13 Word7', 'Word8 Word9 Word10 Word11 Word5']

 Col2
['44912133', '10719', '4.80Z+11', '10731', '4.80F+11']

 Col3
['3,575.00', '1,092.50', '75', '30,980.00', '3,766.80']

 Col4
['Word771 Word772', 'Word711 Word773 Word774', 'Word775 Word776', 'Word777 Word45 Word160 Word45 Word778', 'Word779 Word258 Word780']

 Col5
['8/31/2018', '8/31/2018', '10/31/2018', '01-01-2018', '01-01-2018']

 Col6
['Word560 Word561 Word366 Word562', 'Word563 Word366 Word562', 'Word563 Word564 Word9 Word26', 'Word565 Word566', 'Word567 Word568 Word569 Word570']

 Col7
['Doc1', 'NoDoc', 'NoDoc', 'NoDoc', 'Doc1']


In [66]:
# Create a copy for cleaning

df_clean = df.copy()

In [67]:
# Standardize inconsistent target labels

label_mapping = {
    "category_1": "Category_1",
    "Category2": "Category_2",
    "Category 3": "Category_3",
    "Category _3": "Category_3",
    "Category4": "Category_4",
    "Category 5": "Category_5",
    "Categry_6": "Category_6"
}

df_clean["ClassificationLabel"] = (
    df_clean["ClassificationLabel"].replace(label_mapping)
)

In [68]:
df_clean["ClassificationLabel"].value_counts()

ClassificationLabel
Category_1    5215
Category_2     631
Category_3      23
Category_4      16
Category_6      12
Category_5       2
Name: count, dtype: int64

In [69]:
# Convert Col3 to numeric

df_clean["Col3"] = pd.to_numeric(df_clean["Col3"].str.replace(",", ""),errors="coerce")

In [70]:
df_clean["Col3"].dtype

dtype('float64')

In [71]:
# Convert Col5 to datetime

df_clean["Col5"] = pd.to_datetime(
    df_clean["Col5"],
    errors="coerce",
    format="mixed"
)

In [72]:
print("Minimum date:", df_clean["Col5"].min())
print("Maximum date:", df_clean["Col5"].max())

Minimum date: 2016-10-01 00:00:00
Maximum date: 2018-12-31 00:00:00


In [73]:
# Extract year and month information from the date

df_clean["Year"] = df_clean["Col5"].dt.year
df_clean["Month"] = df_clean["Col5"].dt.month

In [74]:
df_clean["Col7"].value_counts()

Col7
NoDoc    4635
Doc1     1108
Doc3      121
Doc2       35
Name: count, dtype: int64

In [77]:
df_clean["Col2"].nunique()

4818

In [78]:
# Create simple features from Col2

df_clean["Col2_Length"] = df_clean["Col2"].str.len()

df_clean["Col2_Has_Letter"] = (
    df_clean["Col2"].str.contains(r"[A-Za-z]", regex=True)
    .astype(int)
)

df_clean["Col2_Has_Dash"] = (
    df_clean["Col2"].str.contains("-", regex=False)
    .astype(int)
)

In [81]:
df_clean.sample(10)

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,ClassificationLabel,Year,Month,Col2_Length,Col2_Has_Letter,Col2_Has_Dash
2182,Word16 Word17 Word3 Word7,71960,864.00,Word1326 Word1551,2018-10-31,Word563 Word366 Word562,NoDoc,Category_1,2018,10,5,0,0
3800,Word16 Word17 Word18 Word7,TT820,13860.00,Word1501 Word784 Word785,2018-08-31,Word563 Word564 Word9 Word26,NoDoc,Category_1,2018,8,5,1,0
5403,Word16 Word17 Word3 Word7,71215,324.00,Word800 Word804 Word805 Word868,2018-10-31,Word563 Word366 Word562,NoDoc,Category_1,2018,10,5,0,0
3984,Word42 Word43 Word5,7621,930.00,Word1119 Word848 Word1912 Word804 Word914,2018-02-01,Word563 Word580 Word581 Word232,NoDoc,Category_1,2018,2,4,0,0
4351,Word30 Word31,640452,721.25,Word61,2018-02-01,Word563 Word564 Word9 Word26,NoDoc,Category_1,2018,2,6,0,0
3771,Word42 Word43 Word5,8206,2000.00,Word1353 Word1883 Word936 Word1884,2018-07-31,Word576 Word366 Word562,NoDoc,Category_2,2018,7,4,0,0
2337,Word162 Word7,SPX05KNH18,10500.00,Word582 Word1225,2018-08-31,Word567 Word582 Word107,Doc1,Category_1,2018,8,10,1,0
3528,Word19 Word20 Word21 Word22 Word23 Word5,19778,10.08,Word1668 Word1843,2018-10-31,Word563 Word575,NoDoc,Category_2,2018,10,5,0,0
5392,Word16 Word17 Word18 Word7,1633,421.20,Word959 Word784 Word785,2018-01-01,Word563 Word564 Word9 Word26,NoDoc,Category_1,2018,1,4,0,0
1997,Word16 Word17 Word18 Word7,2825,360.00,Word255 Word791,2018-05-31,Word563 Word132 Word610,NoDoc,Category_1,2018,5,4,0,0


In [82]:
df_clean.isnull().sum()

Col1                     0
Col2                     0
Col3                     0
Col4                   153
Col5                     0
Col6                     0
Col7                     0
ClassificationLabel      0
Year                     0
Month                    0
Col2_Length              0
Col2_Has_Letter          0
Col2_Has_Dash            0
dtype: int64